# 1. Indexing

## 1.1 Load the document

In [4]:
from __future__ import annotations

import os
import sys
from pathlib import Path
from dotenv import load_dotenv

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = Path.cwd().parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from arabic_legal_qa.rag.config import EmbeddingConfig, project_root
from arabic_legal_qa.rag.pdf_loader import EgyptianCivilCodeLoader
from arabic_legal_qa.rag.helper import stable_hash


ROOT = project_root(ROOT)
load_dotenv(ROOT / ".env", override=False)
PDF_PATH = ROOT / "data/raw/egyptian_civil_law.pdf"
PROCESSED_DIR = ROOT / "data/processed"

loader = EgyptianCivilCodeLoader(
    pdf_path=PDF_PATH,
    output_dir=PROCESSED_DIR,
    language="bilingual",
)

# One LangChain Document per validated Civil Code article.
documents = loader.load()

sample = next(
    doc for doc in documents
    if doc.metadata["article_number"] == 147 and doc.metadata["language"] == "ar"
)

print({
    "documents": len(documents),
    "articles": len(loader.articles or []),
    "valid": bool(loader.report and loader.report["valid"]),
    "sample_metadata": sample.metadata,
})

print(sample.page_content[:800])

{'documents': 2298, 'articles': 1149, 'valid': True, 'sample_metadata': {'article_number': 147, 'book': 'BOOK I OBLIGATIONS GENERALLY', 'chapter': 'CHAPTER I Sources of Obligations', 'section': 'Section I Contracts', 'topic': '2. The Effects of a Contract', 'is_repealed': False, 'source_page': 16, 'citation': 'Egyptian Civil Code, Article 147', 'language': 'ar', 'chunk_id': '7b8e9510-e54e-551a-b00f-7117b4881a43', 'chunk_position': 0, 'has_source_correction': False}}
( ١ ( العقد شريعة المتعاقدين ، فلا يجوز نقضه ولا تعديله إلا باتفاق
الطرفين ، أو للأسباب التى يقررها القانون.
( ٢ ( ومع ذلك إذا طرأت حوادث استثنائية عامة لم يكن فى الوسع توقعها
وترتب على حدوثها أن تنفيذ الإلتزام التعاقدى ، وان لم يصبح مستحيلا ،
صار مرهقا للمدين بحيث يهدده بخسارة فادحة ، جاز للقاضى تبعا
للظروف وسد الموازنة بين مصلحة الطرفين أن يرد الإلتزام المرهق إلى
الحد المعقول. ويقع باطلا كل اتفاق على خلاف ذلك.


## 1.2 Split the document into chunks

In [5]:
from arabic_legal_qa.rag.llms import load_embedding_bundle
from arabic_legal_qa.rag.chunking import ChunkingConfig, chunk_documents

embedding_config = EmbeddingConfig(
    model_name="intfloat/multilingual-e5-small",
    model_revision=os.getenv("E5_SMALL_REVISION") or None,
)
embedding_bundle = load_embedding_bundle(
    config=embedding_config,
    cache_dir=ROOT / "data/model_cache",
)

corpus_hash = stable_hash([doc.page_content for doc in documents])
# Leave room for E5's passage prefix and special tokens.
chunk_config = ChunkingConfig(embedding_limit=embedding_config.embedding_limit, chunk_tokens=480)
chunks = chunk_documents(
    documents=documents,
    tokenizer=embedding_bundle.tokenizer,
    config=chunk_config,
    corpus_hash=corpus_hash,
)

print({"chunks": len(chunks), "embedding_model": embedding_config.model_name})

{'chunks': 2299, 'embedding_model': 'intfloat/multilingual-e5-small'}


## 1.3 Embed the chunks and store them in a vector database

In [ ]:
from arabic_legal_qa.rag.qdrant_db import QdrantConfig, build_index

qdrant_config = QdrantConfig(
    path=ROOT / "data/vector_store/qdrant",
    exact_search=True
)
index_result = build_index(
    documents=chunks,
    embedding_bundle=embedding_bundle,
    config=qdrant_config,
    corpus_hash=corpus_hash,
)
print({"indexed": index_result["count"], "collection": qdrant_config.collection_name})


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

{'indexed': 2299, 'collection': 'egyptian_civil_code'}


# 2. Retrieval-Augmented Generation (RAG)

## 2.1 Native hybrid retrieval

The query is searched against both E5 dense vectors and Qdrant BM25 sparse vectors, then fused with reciprocal rank fusion.

In [ ]:
from arabic_legal_qa.rag.qdrant_db import hybrid_search

qdrant_client = index_result["client"]
results = hybrid_search(
    query="ما هي آثار العقد؟",
    embedding_bundle=embedding_bundle,
    client=qdrant_client,
    config=qdrant_config,
    k=5,
)
[(doc.metadata.get("article_number"), doc.metadata.get("language"), doc.page_content[:160]) for doc in results]

## 2.2 Inference-time loading

Reopen the persisted collection and model in a fresh process without rebuilding the index.

In [ ]:
from arabic_legal_qa.rag.llms import load_embedding_bundle
from arabic_legal_qa.rag.qdrant_db import QdrantConfig, hybrid_search, open_index

inference_config = EmbeddingConfig(
    model_name="intfloat/multilingual-e5-small",
    model_revision=os.getenv("E5_SMALL_REVISION") or None,
)
inference_bundle = load_embedding_bundle(
    config=inference_config,
    cache_dir=ROOT / "data/model_cache",
)
inference_qdrant_config = QdrantConfig(
    path=ROOT / "data/vector_store/qdrant",
)
inference_client = open_index(inference_qdrant_config)

question = "ما هي آثار العقد؟"
retrieved_docs = hybrid_search(
    query=question,
    embedding_bundle=inference_bundle,
    client=inference_client,
    config=inference_qdrant_config,
    k=5,
)
[(doc.metadata.get("article_number"), doc.metadata.get("language"), doc.page_content[:160]) for doc in retrieved_docs]